# 03 — BM25 Baseline (M2)

Runs `scripts/run_bm25.py`: a from-scratch, transparent Okapi BM25 implementation (`k1=1.2, b=0.75`, frozen literature defaults — `configs/bm25.yaml`, `tuning.enabled: false`) in `src/biomedical_ir/bm25.py`.

In [ ]:
# If running on Colab, clone the repo and install deps. Skipped automatically
# when already inside a local checkout (REPO_ROOT / 'src' already importable).
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('biomedical-hybrid-ir'):
        subprocess.run(['git', 'clone', 'https://github.com/Arungharami/biomedical-hybrid-ir'], check=True)
    os.chdir('biomedical-hybrid-ir')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
print('cwd:', os.getcwd())

In [ ]:
import time
start = time.perf_counter()
result = subprocess.run([sys.executable, 'scripts/run_bm25.py'], cwd=os.getcwd())
print(f'\nexit code: {result.returncode}, elapsed: {time.perf_counter()-start:.1f}s')
assert result.returncode == 0, 'Script failed -- see output above.'

## Real results

In [ ]:
import json
payload = json.load(open('results/metrics/bm25.json'))
print(json.dumps(payload, indent=2)[:3000])

**Note:** BM25 only scores documents sharing ≥1 query term with the query — 25 of 323 test queries (e.g. "deafness", "eggnog", "Fosamax") share zero vocabulary with the corpus after preprocessing and get an empty ranking entirely. This is real BM25 behavior (the classic vocabulary-mismatch problem), not a bug — see `docs/models.md` §M5 and `results/error-analysis/error_analysis.md`.

**M7 finding:** TF-IDF significantly *outperforms* BM25 here (paired bootstrap, p<0.03 on P@10/Recall@100/nDCG@10) — the opposite of the textbook expectation. See `results/tables/statistical_tests.md`.